In [ ]:
"""# Colab setup — CPU is fine for this step and for later fine-tuning, given the dataset size
!pip install -q datasets transformers scikit-learn

import pandas as pd
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from transformers import AutoTokenizer, AutoModel

# ---------------------------------------------------------
# 1. Load chart_of_accounts subset from each country repo
# ---------------------------------------------------------
countries = {
    "us": "mindweave/accounting-ledger-us",
    "uk": "mindweave/accounting-ledger-uk",
    "au": "mindweave/accounting-ledger-au",
}

country_dfs = {}
for country, repo_id in countries.items():
    ds = load_dataset(repo_id, "chart_of_accounts")   # single "train" split
    df = ds["train"].to_pandas()
    df["country"] = country
    country_dfs[country] = df
    print(f"{country}: {len(df)} rows")

# ---------------------------------------------------------
# 2. Split each country's data separately (stratified on account_type,
#    since sub_type has too few examples per class to stratify on reliably),
#    then combine the per-country train/test splits
# ---------------------------------------------------------
train_parts, test_parts = [], []

for country, df in country_dfs.items():
    train_c, test_c = train_test_split(
        df,
        test_size=0.2,
        stratify=df["account_type"],
        random_state=42,
    )
    train_parts.append(train_c)
    test_parts.append(test_c)

train_df = pd.concat(train_parts, ignore_index=True)
test_df = pd.concat(test_parts, ignore_index=True)

print(f"\nCombined train: {len(train_df)} rows")
print(f"Combined test:  {len(test_df)} rows")
print("\nTrain country breakdown:\n", train_df["country"].value_counts())
print("\nTest country breakdown:\n", test_df["country"].value_counts())

# ---------------------------------------------------------
# 3. Encode labels (fit on the full combined data since it's a
#    small, fixed, closed set of categories across all 3 countries)
# ---------------------------------------------------------
full_df = pd.concat([train_df, test_df], ignore_index=True)

type_encoder = LabelEncoder().fit(full_df["account_type"])
subtype_encoder = LabelEncoder().fit(full_df["sub_type"])

train_df["account_type_label"] = type_encoder.transform(train_df["account_type"])
train_df["sub_type_label"] = subtype_encoder.transform(train_df["sub_type"])
test_df["account_type_label"] = type_encoder.transform(test_df["account_type"])
test_df["sub_type_label"] = subtype_encoder.transform(test_df["sub_type"])

print(f"\naccount_type classes ({len(type_encoder.classes_)}):", list(type_encoder.classes_))
print(f"sub_type classes ({len(subtype_encoder.classes_)}):", list(subtype_encoder.classes_))

# ---------------------------------------------------------
# 4. Save combined splits to a "train" folder and "test" folder
# ---------------------------------------------------------
import os
os.makedirs("/content/data/train", exist_ok=True)
os.makedirs("/content/data/test", exist_ok=True)

train_df.to_csv("/content/data/train/train.csv", index=False)
test_df.to_csv("/content/data/test/test.csv", index=False)"""

In [ ]:
!pip install -q -U google-genai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 260.0/260.0 kB 15.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.57.1 which is incompatible.


In [ ]:
!pip install -q transformers scikit-learn pandas torch openpyxl

In [ ]:
!pip install deep-translator langdetect -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 15.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 1.4 MB/s eta 0:00:00


In [ ]:
import shutil
import os

# Specify the path to your folder
folder_path = '/content/data'

# Check if the folder exists before deleting
if os.path.exists(folder_path):
    shutil.rmtree(folder_path)
    print(f"Successfully deleted: {folder_path}")
else:
    print("The folder does not exist.")

The folder does not exist.


# Data Preparation

In [ ]:
import time
start_time = time.time()

In [ ]:
import os, subprocess, io, re
import pandas as pd
from sklearn.model_selection import train_test_split

os.makedirs("/content/data", exist_ok=True)

# ---------------------------------------------------------
# 1. Shallow, blob-less clone — metadata only, no bulk downloads.
#    Also sidesteps the old bug: blob URLs return HTML, not CSV.
# ---------------------------------------------------------
REPO_DIR = "/content/odoo_repo"
if not os.path.exists(REPO_DIR):
    subprocess.run(
        ["git", "clone", "--depth", "1", "--filter=blob:none", "--no-checkout",
         "-b", "17.0", "https://github.com/odoo/odoo.git", REPO_DIR],
        check=True
    )

def git(*args):
    return subprocess.run(["git", "-C", REPO_DIR] + list(args),
                           capture_output=True, text=True)

# ---------------------------------------------------------
# 2. Discover ALL l10n_XX modules and locate their
#    Chart-of-Accounts CSV files regardless of folder layout.
# ---------------------------------------------------------

folders = git(
    "ls-tree", "-d", "--name-only", "HEAD:addons"
).stdout.splitlines()

two_letter_codes = sorted(
    m.group(1)
    for f in folders
    if (m := re.match(r"^l10n_([a-z]{2})$", f))
)

print(f"Total l10n_XX modules: {len(two_letter_codes)}")


# ---------------------------------------------------------
# 3. Search each localization module recursively for CSVs
#    that appear to contain Chart-of-Accounts data.
# ---------------------------------------------------------

country_files = {}
no_coa = []

for code in two_letter_codes:

    module = f"addons/l10n_{code}"

    # Get every file in this localization module
    result = git("ls-tree", "-r", "--name-only", f"HEAD:{module}")

    if result.returncode != 0:
        no_coa.append(code)
        continue

    files = result.stdout.splitlines()

    # Find CSV files that look like account / COA files
    csv_candidates = [
        f for f in files
        if f.lower().endswith(".csv")
        and (
            "account" in os.path.basename(f).lower()
            or "coa" in os.path.basename(f).lower()
        )
    ]

    found = False

    for relative_path in csv_candidates:

        full_path = f"{module}/{relative_path}"

        result = git("show", f"HEAD:{full_path}")

        if result.returncode != 0:
            continue

        try:
            df = pd.read_csv(io.StringIO(result.stdout), nrows=5)
        except Exception:
            continue

        # Accept files containing the basic COA fields
        columns_lower = {c.lower(): c for c in df.columns}

        has_name = "name" in columns_lower
        has_type = "account_type" in columns_lower

        if has_name and has_type:
            country_files[code] = relative_path
            found = True
            break

    if not found:
        no_coa.append(code)


# ---------------------------------------------------------
# 4. Report exactly what was discovered
# ---------------------------------------------------------

print(f"\nCountries with usable COA CSV: {len(country_files)}")
print(f"Countries without usable COA CSV: {len(no_coa)}")

print("\nCOA files found:")
for code, path in country_files.items():
    print(f"{code}: {path}")

print("\nNo usable COA CSV found:")
print(no_coa)

# ---------------------------------------------------------
# 5.. Load + validate each candidate (required columns + usable rows),
# ---------------------------------------------------------
REQUIRED_COLS = ["id", "name", "account_type"]
#TARGET_COUNTRIES = 80

country_dfs = {}

for code, relative_path in country_files.items():

    path = f"addons/l10n_{code}/{relative_path}"

    result = git("show", f"HEAD:{path}")

    if result.returncode != 0:
        print(f"{code}: FAILED to fetch {relative_path}")
        continue

    try:
        df = pd.read_csv(io.StringIO(result.stdout))
    except Exception as e:
        print(f"{code}: FAILED to parse ({e})")
        continue

    # Handle column names
    if not all(c in df.columns for c in ["id", "name", "account_type"]):
        print(f"{code}: missing required columns")
        continue

    df = df[["id", "name", "account_type"]].rename(
        columns={
            "id": "account_id",
            "name": "account_name"
        }
    )

    df = df.dropna(subset=["account_name", "account_type"])

    if df.empty:
        continue

    df["country"] = code
    country_dfs[code] = df

print(f"\nUsing {len(country_dfs)} countries: {list(country_dfs.keys())}")

combined_df = pd.concat(country_dfs.values(), ignore_index=True)
print(f"Combined (pre-cutoff): {len(combined_df)} rows")

Total l10n_XX modules: 104

Countries with usable COA CSV: 85
Countries without usable COA CSV: 19

COA files found:
ae: data/template/account.account-ae.csv
ar: data/template/account.account-ar_base.csv
at: data/template/account.account-at.csv
au: data/template/account.account-au.csv
bd: data/template/account.account-bd.csv
be: data/template/account.account-be.csv
bg: data/template/account.account-bg.csv
bh: data/template/account.account-bh.csv
bo: data/template/account.account-bo.csv
br: data/template/account.account-br.csv
ca: data/template/account.account-ca_2023.csv
ch: data/template/account.account-ch.csv
cl: data/template/account.account-cl.csv
cn: data/template/account.account-cn.csv
co: data/template/account.account-co.csv
cr: data/template/account.account-cr.csv
cy: data/template/account.account-cy.csv
cz: data/template/account.account-cz.csv
de: data/template/account.account-de_skr03.csv
dk: data/template/account.account-dk.csv
do: data/template/account.account-do.csv
dz: da

In [ ]:
from deep_translator import GoogleTranslator
from langdetect import detect, LangDetectException, DetectorFactory
import time

DetectorFactory.seed = 42

def is_english(text):
    try:
        return detect(text) == "en"
    except LangDetectException:
        return True  # too short/ambiguous to detect — leave as-is

def translate_account_name(name, retries=3):
    for attempt in range(retries):
        try:
            return GoogleTranslator(source="auto", target="en").translate(name)
        except Exception as e:
            if attempt == retries - 1:
                print(f"Translation failed for '{name}': {e}")
                return name  # keep original, flag for manual review
            time.sleep(1)

# run right after building combined_df from Odoo, BEFORE dedup
combined_df["account_name_original"] = combined_df["account_name"]
non_en_mask = combined_df["account_name"].astype(str).apply(lambda x: not is_english(x))
print(f"Translating {non_en_mask.sum()} non-English account names.")
combined_df.loc[non_en_mask, "account_name"] = combined_df.loc[non_en_mask, "account_name"].apply(translate_account_name)


# takes 10 mins

Translating 5065 non-English account names.


In [ ]:
# ---------------------------------------------------------
# dedup pass across the combined dataset
# (per-batch dedup later in synthetic data prevents most duplicates via
# existing_names / existing_name_to_type; this is a cleanup pass)
# ---------------------------------------------------------

before = len(combined_df)
combined_df = combined_df.drop_duplicates(subset=["account_name", "account_type"]).reset_index(drop=True)
print(f"\nDropped {before - len(combined_df)} duplicate rows from final combined dataset.")

print("\nCounts per class after original duplicate removal:")
print(combined_df["account_type"].value_counts().sort_index())


Dropped 4104 duplicate rows from final combined dataset.

Counts per class after original duplicate removal:
account_type
asset_cash                276
asset_current            3028
asset_fixed              1389
asset_non_current        1553
asset_prepayments         130
asset_receivable          538
equity                    991
equity_unaffected          29
expense                  5051
expense_depreciation      375
expense_direct_cost       408
income                   1558
income_other              856
liability_credit_card       1
liability_current        2511
liability_non_current     722
liability_payable         428
off_balance               267
Name: count, dtype: int64


Synthetic Data

In [ ]:
TARGET_CLASSES = [
    "asset_current",
    "asset_fixed",
    "asset_non_current",
    "asset_receivable",
    "asset_cash",
    "asset_prepayments",

    "liability_current",
    "liability_non_current",
    "liability_payable",
    "liability_credit_card",

    "equity",
    "equity_unaffected",

    "income",
    "income_other",

    "expense",
    "expense_direct_cost",
    "expense_depreciation",

    "off_balance",
]

In [ ]:
COMBINED_FILE = "/content/data/combined_with_synthetic.csv"

In [ ]:
import requests
os.environ["GEMINI_API_KEY"] = "placeholder"    # "YOUR_API_KEY_HERE"
os.environ["CF_ACCOUNT_ID"] = "placeholder"
os.environ["CF_API_TOKEN"] = "placeholder"

In [ ]:
"""
# ---------------------------------------------------------
# 3b. SYNTHETIC DATA GENERATION
#     Bring every existing account_type to at least 1000 rows.
#     GEMINI SYNTHETIC COA DATA GENERATION
# ---------------------------------------------------------

from google import genai
from google.genai import types
import json
import time
import random
import numpy as np
from pathlib import Path

# ---------------------------------------------------------
# Configuration
# ---------------------------------------------------------

MODEL = "gemini-2.5-flash"
BATCH_SIZE = 100
EXAMPLES_PER_PROMPT = 10
SYNTHETIC_FILE = "/content/data/synthetic_coa.csv"
COMBINED_FILE = "/content/data/combined_with_synthetic.csv"
REQUEST_DELAY = 2

client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

TARGET_PER_CLASS = 600

# ---------------------------------------------------------
# Persistent synthetic-data file
# ---------------------------------------------------------

if os.path.exists(SYNTHETIC_FILE):
    synthetic_df = pd.read_csv(SYNTHETIC_FILE)
    print(f"Loaded {len(synthetic_df)} previously generated synthetic rows.")

    expected_cols = ["account_id", "account_name", "account_type", "country", "is_synthetic"]

    # backfill is_synthetic for files saved before this column existed
    if "is_synthetic" not in synthetic_df.columns:
        synthetic_df["is_synthetic"] = True
    else:
        synthetic_df["is_synthetic"] = True  # every row in this file is synthetic by definition

    synthetic_df = synthetic_df[[c for c in expected_cols if c in synthetic_df.columns]]

    combined_df = pd.concat([combined_df, synthetic_df], ignore_index=True)
    combined_df = combined_df.drop_duplicates(subset=["account_name", "account_type"]).reset_index(drop=True)

else:
    synthetic_df = pd.DataFrame(columns=["account_id", "account_name", "account_type", "country", "is_synthetic"])

# Flag every pre-existing row in combined_df that ISN'T synthetic as real.
if "is_synthetic" not in combined_df.columns:
    combined_df["is_synthetic"] = False
else:
    combined_df["is_synthetic"] = combined_df["is_synthetic"].fillna(False)

# ---------------------------------------------------------
# Utility: calculate current deficits
# ---------------------------------------------------------

def get_deficits(df):
    counts = df["account_type"].value_counts()
    deficits = {}
    for account_type in TARGET_CLASSES:
        count = counts.get(account_type, 0)
        if count < TARGET_PER_CLASS:
            deficits[account_type] = TARGET_PER_CLASS - count
    return dict(sorted(deficits.items(), key=lambda x: x[1], reverse=True))

# ---------------------------------------------------------
# Utility: get real examples for a class
# ---------------------------------------------------------

def get_examples(df, account_type, n=10):
    # deliberately sample from REAL rows only — synthetic examples
    # feeding back into the prompt would compound drift over time
    real_df = df[df["is_synthetic"] == False]
    examples = (
        real_df[real_df["account_type"] == account_type]["account_name"]
        .dropna().astype(str).drop_duplicates().tolist()
    )
    if len(examples) <= n:
        return examples
    return random.sample(examples, n)

# ---------------------------------------------------------
# Utility: generate a stable synthetic ID
# ---------------------------------------------------------

def make_synthetic_id(account_type, account_name):
    import hashlib
    key = f"{account_type}::{account_name}"
    digest = hashlib.md5(key.encode("utf-8")).hexdigest()[:12]
    return f"synthetic_{digest}"

# ---------------------------------------------------------
# Gemini generation function
# (response_schema removed — Gemini rejects tools + structured
#  output together with 400 INVALID_ARGUMENT; google_search kept
#  so names are grounded in real terminology, not just paraphrased
#  from the examples)
# ---------------------------------------------------------

def generate_account_names(account_type, examples, existing_names, requested_count):
    examples_text = "\n".join(f"- {x}" for x in examples)

    existing_sample = list(existing_names)
    if len(existing_sample) > 1000:
        existing_sample = random.sample(existing_sample, 1000)
    existing_text = "\n".join(f"- {x}" for x in existing_sample)

    prompt = f"""
You are generating synthetic Chart-of-Accounts account names
for a financial-accounting machine-learning dataset.

TARGET ACCOUNT TYPE:
{account_type}

TASK:
Generate exactly {requested_count} NEW and UNIQUE account names
that legitimately belong to the target account type.

IMPORTANT:
The account names you generate must be realistic accounting
account names that could plausibly occur in real-world
Chart-of-Accounts systems.

You have been given real examples from the dataset below.
These examples are ONLY STYLE AND SEMANTIC REFERENCES.

REAL EXAMPLES:
{examples_text}

DO NOT copy the examples.
DO NOT make trivial variations of the examples.
DO NOT simply add words such as "New", "Other", "Additional",
"1", "2", etc. to an existing account name.

Each generated name must represent a genuinely plausible
account that could independently appear in a real company's
Chart of Accounts.

Use appropriate accounting terminology.

Use Google Search to ground your understanding of the target
account type — look up real-world Chart-of-Accounts examples,
ERP systems (e.g. Odoo, QuickBooks, SAP, Xero), and standard
accounting/bookkeeping terminology for this account type.
Prefer authoritative accounting, ERP, or financial sources.

STRICT SEMANTIC REQUIREMENT:
Every generated account name must belong to exactly:

{account_type}

Do not generate names belonging to another account type.
An account name must never plausibly belong to more than
one account type — pick names that are unambiguous.

DIVERSITY:
Generate diverse but realistic names.
Avoid producing many near-synonyms of the same account.
Avoid excessive use of generic words such as "Other".

UNIQUENESS:
Every generated name must be unique within this response.
Also avoid names already present in the existing dataset.

EXISTING DATASET NAMES TO AVOID:
{existing_text}

OUTPUT:
Return ONLY a JSON object with exactly this structure, and
nothing else — no markdown, no code fences, no commentary
before or after it, no citations inside the JSON:

{{
  "account_names": ["Account name 1", "Account name 2"]
}}
"""

    response = client.models.generate_content(
        model=MODEL,
        contents=prompt,
        config=types.GenerateContentConfig(
            tools=[types.Tool(google_search=types.GoogleSearch())]
        )
    )

    raw = response.text.strip()

    # strip accidental code fences even though the prompt forbids them
    if raw.startswith("```"):
        raw = raw.strip("`")
        raw = raw.split("\n", 1)[1] if "\n" in raw else raw
        if raw.lower().startswith("json"):
            raw = raw.split("\n", 1)[1]

    start, end = raw.find("{"), raw.rfind("}")
    if start == -1 or end == -1:
        raise ValueError(f"No JSON object found in Gemini response: {raw[:200]}")

    data = json.loads(raw[start:end + 1])
    return data.get("account_names", [])

def generate_account_names_cloudflare(account_type, examples, existing_names, requested_count):
    CF_ACCOUNT_ID = os.environ["CF_ACCOUNT_ID"]
    CF_API_TOKEN = os.environ["CF_API_TOKEN"]
    CF_MODEL = "@cf/meta/llama-3.3-70b-instruct-fp8-fast"

    examples_text = "\n".join(f"- {x}" for x in examples)
    existing_sample = random.sample(list(existing_names), min(1000, len(existing_names)))
    existing_text = "\n".join(f"- {x}" for x in existing_sample)

    prompt = f"""
    You are generating synthetic Chart-of-Accounts account names
    for a financial-accounting machine-learning dataset.

    TARGET ACCOUNT TYPE:
    {account_type}

    TASK:
    Generate exactly {requested_count} NEW and UNIQUE account names
    that legitimately belong to the target account type.

    IMPORTANT:
    The account names you generate must be realistic accounting
    account names that could plausibly occur in real-world
    Chart-of-Accounts systems.

    You have been given real examples from the dataset below.
    These examples are ONLY STYLE AND SEMANTIC REFERENCES.

    REAL EXAMPLES:
    {examples_text}

    DO NOT copy the examples.
    DO NOT make trivial variations of the examples.
    DO NOT simply add words such as "New", "Other", "Additional",
    "1", "2", etc. to an existing account name.

    Each generated name must represent a genuinely plausible
    account that could independently appear in a real company's
    Chart of Accounts.

    Use appropriate accounting terminology.

    Use Google Search to ground your understanding of the target
    account type — look up real-world Chart-of-Accounts examples,
    ERP systems (e.g. Odoo, QuickBooks, SAP, Xero), and standard
    accounting/bookkeeping terminology for this account type.
    Prefer authoritative accounting, ERP, or financial sources.

    STRICT SEMANTIC REQUIREMENT:
    Every generated account name must belong to exactly:

    {account_type}

    Do not generate names belonging to another account type.
    An account name must never plausibly belong to more than
    one account type — pick names that are unambiguous.

    DIVERSITY:
    Generate diverse but realistic names.
    Avoid producing many near-synonyms of the same account.
    Avoid excessive use of generic words such as "Other".

    UNIQUENESS:
    Every generated name must be unique within this response.
    Also avoid names already present in the existing dataset.

    EXISTING DATASET NAMES TO AVOID:
    {existing_text}

    OUTPUT:
    Return ONLY a JSON object with exactly this structure, and
    nothing else — no markdown, no code fences, no commentary
    before or after it, no citations inside the JSON:

    {{
      "account_names": ["Account name 1", "Account name 2"]
    }}
    """

    resp = requests.post(
        f"https://api.cloudflare.com/client/v4/accounts/{CF_ACCOUNT_ID}/ai/run/{CF_MODEL}",
        headers={"Authorization": f"Bearer {CF_API_TOKEN}"},
        json={"messages": [{"role": "user", "content": prompt}]},
        timeout=60
    )
    resp.raise_for_status()
    raw = resp.json()["result"]["response"].strip()

    if raw.startswith("```"):
        raw = raw.strip("`")
        raw = raw.split("\n", 1)[1] if "\n" in raw else raw
        if raw.lower().startswith("json"):
            raw = raw.split("\n", 1)[1]

    start, end = raw.find("{"), raw.rfind("}")
    if start == -1 or end == -1:
        raise ValueError(f"No JSON object found in Cloudflare response: {raw[:200]}")

    return json.loads(raw[start:end + 1]).get("account_names", [])


def generate_account_names_with_fallback(account_type, examples, existing_names, requested_count):
    backoff = [5, 15, 30]
    for attempt in range(3):
        try:
            return generate_account_names(account_type, examples, existing_names, requested_count)
        except Exception as e:
            print(f"Gemini attempt {attempt+1}/3 failed: {e}")
            if attempt < 2:
                time.sleep(backoff[attempt])

    print("Gemini exhausted after 3 attempts — falling back to Cloudflare (Llama 3.3 70B).")

    cf_backoff = [3, 8]
    for attempt in range(2):
        try:
            return generate_account_names_cloudflare(account_type, examples, existing_names, requested_count)
        except Exception as e:
            print(f"Cloudflare attempt {attempt+1}/2 failed: {e}")
            if attempt < 1:
                time.sleep(cf_backoff[attempt])

    raise RuntimeError(f"Both Gemini and Cloudflare failed for {account_type}")
    """

In [ ]:
"""
# ---------------------------------------------------------
# Main generation loop
# ---------------------------------------------------------

print("\n==============================================")
print("STARTING SYNTHETIC DATA GENERATION")
print("==============================================\n")

fail_counts = {}

while True:

    deficits = get_deficits(combined_df)

    if not deficits:
        print("\nAll account types have reached 1000 instances.")
        break

    account_type, deficit = next(iter(deficits.items()))
    current_count = combined_df["account_type"].eq(account_type).sum()
    batch_size = min(BATCH_SIZE, deficit)

    print("\n----------------------------------------------")
    print(f"Account type : {account_type}")
    print(f"Current count: {current_count}")
    print(f"Remaining    : {deficit}")
    print(f"This batch   : {batch_size}")
    print("----------------------------------------------")

    examples = get_examples(combined_df, account_type, EXAMPLES_PER_PROMPT)
    print("\nExamples supplied to Gemini:")
    for x in examples:
        print(f"  {x}")

    existing_names = set(
        combined_df["account_name"].dropna().astype(str).str.strip().str.lower()
    )

    try:
        generated_names = generate_account_names_with_fallback(
    account_type=account_type,
    examples=examples,
    existing_names=existing_names,
    requested_count=batch_size
)
    except Exception as e:
        fail_counts[account_type] = fail_counts.get(account_type, 0) + 1
        print(f"\n{account_type} failed ({fail_counts[account_type]}x): {e}")
        if fail_counts[account_type] >= 3:
            print(f"Giving up on {account_type} after 3 full failures — moving on.")
            #TARGET_CLASSES.remove(account_type)  # or maintain a skip-set instead
        continue

    # -------------------------------------------------------------
    # Local validation — Python is the final gatekeeper, not Gemini.
    # Also catch and flag any name Gemini reused across account
    # types, since a name should never legitimately belong to two
    # classes.
    # -------------------------------------------------------------

    existing_name_to_type = (
        combined_df.dropna(subset=["account_name", "account_type"])
        .assign(_norm=lambda d: d["account_name"].astype(str).str.strip().str.lower())
        .drop_duplicates(subset="_norm")
        .set_index("_norm")["account_type"]
        .to_dict()
    )

    accepted_names = []
    seen_this_batch = set()
    cross_type_flags = []

    for name in generated_names:
        if not isinstance(name, str):
            continue

        name = name.strip()
        if not name:
            continue

        normalized = name.lower()

        if normalized in existing_name_to_type:
            existing_type = existing_name_to_type[normalized]
            if existing_type != account_type:
                cross_type_flags.append((name, account_type, existing_type))
            continue  # duplicate against existing dataset either way

        if normalized in seen_this_batch:
            continue  # duplicate within this batch

        seen_this_batch.add(normalized)
        accepted_names.append(name)

    # isme confusing account name tou dono mein se remove krdena chahiye na..?
    if cross_type_flags:
        print("\n⚠️  Gemini returned names already claimed by a different account_type — dropped:")
        for name, requested, existing in cross_type_flags:
            print(f"  '{name}' — requested as {requested}, already exists as {existing}")

    print(f"\nGemini returned: {len(generated_names)}")
    print(f"Accepted locally: {len(accepted_names)}")

    if not accepted_names:
        print("\nNo usable names were returned.\nStopping to avoid an infinite loop.")
        break

    # -------------------------------------------------------------
    # Convert generated names into dataset rows.
    # -------------------------------------------------------------

    new_rows = []
    for name in accepted_names:
        new_rows.append({
            "account_id": make_synthetic_id(account_type, name),
            "account_name": name,
            "account_type": account_type,
            "country": "synthetic",
            "is_synthetic": True
        })

    new_df = pd.DataFrame(new_rows)

    combined_df = pd.concat([combined_df, new_df], ignore_index=True)

    synthetic_df = pd.concat([synthetic_df, new_df], ignore_index=True)
    synthetic_df = synthetic_df.drop_duplicates(subset=["account_name", "account_type"]).reset_index(drop=True)
    synthetic_df.to_csv(SYNTHETIC_FILE, index=False)

    combined_df.to_csv(COMBINED_FILE, index=False)

    new_count = combined_df["account_type"].eq(account_type).sum()
    remaining = max(0, TARGET_PER_CLASS - new_count)

    print(f"\n{account_type}: {new_count}/1000 | remaining: {remaining}")
    print(f"Saved synthetic data to:\n{SYNTHETIC_FILE}")
    print(f"Saved combined data to:\n{COMBINED_FILE}")

    time.sleep(REQUEST_DELAY)

# ---------------------------------------------------------
# Final report
# ---------------------------------------------------------

print("\n==============================================")
print("FINAL SYNTHETIC DATA STATUS")
print("==============================================")

final_counts = combined_df["account_type"].value_counts().sort_values()
print(final_counts)

remaining_deficits = get_deficits(combined_df)
if remaining_deficits:
    print("\nRemaining deficits:")
    for cls, deficit in remaining_deficits.items():
        print(f"{cls}: {deficit}")
else:
    print("\nSUCCESS: Every class has at least 1000 instances.")
    """


STARTING SYNTHETIC DATA GENERATION


----------------------------------------------
Account type : liability_credit_card
Current count: 1
Remaining    : 599
This batch   : 100
----------------------------------------------

Examples supplied to Gemini:
  Company Credit Card

Gemini returned: 100
Accepted locally: 100

liability_credit_card: 101/1000 | remaining: 499
Saved synthetic data to:
/content/data/synthetic_coa.csv
Saved combined data to:
/content/data/combined_with_synthetic.csv

----------------------------------------------
Account type : equity_unaffected
Current count: 29
Remaining    : 571
This batch   : 100
----------------------------------------------

Examples supplied to Gemini:
  Result for the financial year
  Net income for the period
  Retained earnings for the reporting period
  Profit/loss for the period
  Profit and loss account
  Retained earnings for the current year
  Net income
  Acknowledged Net Profit (Loss) of The Accounting Year in The Profit (Loss) Re

KeyboardInterrupt: 

In [ ]:
"""import google.generativeai as genai

# --- swap to a model that still has quota ---
FALLBACK_MODEL = "gemini-3.6-flash"  # change to whatever model has quota left
_model = genai.GenerativeModel(FALLBACK_MODEL)

def generate_account_names_temp(account_type, examples, existing_names, requested_count):
    examples_text = "\n".join(f"- {e}" for e in examples)
    prompt = f"""Generate {requested_count} realistic chart-of-accounts account names
for the account_type "{account_type}".

Examples of existing names in this category:
{examples_text}

Rules:
- Return ONLY a plain list, one name per line, no numbering, no extra text.
- Do not repeat any of the example names.
- Names must be plausible real-world accounting ledger entries.
"""
    response = _model.generate_content(prompt)
    lines = [l.strip("-• \t") for l in response.text.splitlines()]
    return [l for l in lines if l]

# --- same loop as before, but using generate_account_names_temp ---
while True:
    deficits = get_deficits(combined_df)
    if not deficits:
        print("\nAll account types have reached target.")
        break

    account_type, deficit = next(iter(deficits.items()))
    current_count = combined_df["account_type"].eq(account_type).sum()
    batch_size = min(BATCH_SIZE, deficit)

    print(f"\n{account_type} | current: {current_count} | remaining: {deficit} | batch: {batch_size}")

    examples = get_examples(combined_df, account_type, EXAMPLES_PER_PROMPT)
    existing_names = set(
        combined_df["account_name"].dropna().astype(str).str.strip().str.lower()
    )

    try:
        generated_names = generate_account_names_temp(
            account_type, examples, existing_names, batch_size
        )
    except Exception as e:
        print(f"{account_type} failed: {e}")
        time.sleep(REQUEST_DELAY)
        continue

    existing_name_to_type = (
        combined_df.dropna(subset=["account_name", "account_type"])
        .assign(_norm=lambda d: d["account_name"].astype(str).str.strip().str.lower())
        .drop_duplicates(subset="_norm")
        .set_index("_norm")["account_type"]
        .to_dict()
    )

    accepted_names = []
    seen_this_batch = set()

    for name in generated_names:
        if not isinstance(name, str):
            continue
        name = name.strip()
        if not name:
            continue
        normalized = name.lower()
        if normalized in existing_name_to_type or normalized in seen_this_batch:
            continue
        seen_this_batch.add(normalized)
        accepted_names.append(name)

    print(f"Generated: {len(generated_names)} | Accepted: {len(accepted_names)}")

    if not accepted_names:
        time.sleep(REQUEST_DELAY)
        continue

    new_rows = [
        {
            "account_id": make_synthetic_id(account_type, name),
            "account_name": name,
            "account_type": account_type,
            "country": "synthetic",
            "is_synthetic": True
        }
        for name in accepted_names
    ]
    new_df = pd.DataFrame(new_rows)

    combined_df = pd.concat([combined_df, new_df], ignore_index=True)
    synthetic_df = pd.concat([synthetic_df, new_df], ignore_index=True)
    synthetic_df = synthetic_df.drop_duplicates(subset=["account_name", "account_type"]).reset_index(drop=True)

    synthetic_df.to_csv(SYNTHETIC_FILE, index=False)
    combined_df.to_csv(COMBINED_FILE, index=False)

    new_count = combined_df["account_type"].eq(account_type).sum()
    print(f"{account_type}: {new_count}/{TARGET_PER_CLASS}")

    time.sleep(REQUEST_DELAY)

print("\nDone.")
print(combined_df["account_type"].value_counts().sort_values())"""


liability_payable | current: 428 | remaining: 172 | batch: 100
Generated: 100 | Accepted: 96
liability_payable: 524/600

expense_depreciation | current: 475 | remaining: 125 | batch: 100
Generated: 100 | Accepted: 100
expense_depreciation: 575/600

off_balance | current: 475 | remaining: 125 | batch: 100
Generated: 100 | Accepted: 100
off_balance: 575/600

asset_cash | current: 476 | remaining: 124 | batch: 100
Generated: 100 | Accepted: 100
asset_cash: 576/600

liability_credit_card | current: 480 | remaining: 120 | batch: 100
Generated: 100 | Accepted: 100
liability_credit_card: 580/600

asset_prepayments | current: 503 | remaining: 97 | batch: 97
Generated: 97 | Accepted: 87
asset_prepayments: 590/600

expense_direct_cost | current: 511 | remaining: 89 | batch: 89
Generated: 89 | Accepted: 88
expense_direct_cost: 599/600

equity_unaffected | current: 512 | remaining: 88 | batch: 88
Generated: 88 | Accepted: 80
equity_unaffected: 592/600

liability_payable | current: 524 | remaining

KeyboardInterrupt: 

In [ ]:
"""# ---------------------------------------------------------
# 3b. SYNTHETIC DATA GENERATION
#     Bring every existing account_type to at least 1000 rows.
#
#     IMPORTANT:
#     - Original Odoo rows are never modified.
#     - Synthetic rows are generated ONLY for existing classes
#       that have fewer than 1000 examples.
#     - Generation is based on real account names already present
#       in that class.
#     - Synthetic rows are created BEFORE train/test splitting.
# ---------------------------------------------------------

TARGET_INSTANCES = 1000
SYNTHETIC_RANDOM_STATE = 42

import random
import numpy as np

random.seed(SYNTHETIC_RANDOM_STATE)
np.random.seed(SYNTHETIC_RANDOM_STATE)


def generate_synthetic_account_name(account_name, account_type):
    """
    Generate a new plausible COA account name from an existing
    account name.

    The method uses controlled linguistic transformations rather
    than copying the original row verbatim.
    """

    name = str(account_name).strip()

    # ---------------------------------------------------------
    # Common accounting-language transformations.
    # These are intentionally conservative.
    # ---------------------------------------------------------

    replacements = [
        ("Account", "A/c"),
        ("A/c", "Account"),
        ("Accounts", "A/cs"),
        ("Receivable", "Receivables"),
        ("Payable", "Payables"),
        ("Expense", "Expenses"),
        ("Expenses", "Expense"),
        ("Income", "Revenue"),
        ("Revenue", "Income"),
        ("Cost", "Costs"),
        ("Costs", "Cost"),
        ("Provision", "Provision for"),
    ]

    # Randomly apply one transformation if applicable
    possible = [
        (old, new)
        for old, new in replacements
        if old.lower() in name.lower()
    ]

    if possible and random.random() < 0.45:
        old, new = random.choice(possible)

        pattern = re.compile(re.escape(old), re.IGNORECASE)
        name = pattern.sub(new, name, count=1)

    # ---------------------------------------------------------
    # Conservative additions based on accounting terminology.
    # ---------------------------------------------------------

    prefixes = {
        "asset_current": [
            "Current", "Short-Term", "Operating"
        ],
        "asset_fixed": [
            "Fixed", "Property", "Operating"
        ],
        "asset_non_current": [
            "Long-Term", "Non-Current", "Deferred"
        ],
        "asset_receivable": [
            "Trade", "Customer", "Other"
        ],
        "asset_cash": [
            "Main", "Operating", "Petty", "Bank"
        ],
        "asset_prepayments": [
            "Prepaid", "Advance"
        ],
        "liability_current": [
            "Current", "Short-Term", "Accrued"
        ],
        "liability_non_current": [
            "Long-Term", "Non-Current", "Deferred"
        ],
        "liability_payable": [
            "Trade", "Supplier", "Other"
        ],
        "liability_credit_card": [
            "Corporate", "Business", "Company"
        ],
        "equity": [
            "Shareholders'", "Owners'", "Partners'"
        ],
        "equity_unaffected": [
            "Retained", "Opening", "Prior-Year"
        ],
        "income": [
            "Operating", "Sales", "Service"
        ],
        "income_other": [
            "Other", "Non-Operating", "Miscellaneous"
        ],
        "expense": [
            "General", "Administrative", "Operating"
        ],
        "expense_direct_cost": [
            "Direct", "Product", "Production"
        ],
        "expense_depreciation": [
            "Accumulated", "Depreciation"
        ],
        "off_balance": [
            "Memorandum", "Statistical", "Contingent"
        ]
    }

    # Only add a prefix when it does not already appear.
    if account_type in prefixes and random.random() < 0.35:
        prefix = random.choice(prefixes[account_type])

        if prefix.lower() not in name.lower():
            name = f"{prefix} {name}"

    # ---------------------------------------------------------
    # Remove accidental duplicated whitespace.
    # ---------------------------------------------------------

    name = re.sub(r"\s+", " ", name).strip()

    return name


# ---------------------------------------------------------
# Generate exactly the deficit for each class.
# ---------------------------------------------------------

synthetic_rows = []

class_counts = combined_df["account_type"].value_counts()

for account_type, count in class_counts.items():

    deficit = TARGET_INSTANCES - count

    if deficit <= 0:
        continue

    class_data = combined_df[
        combined_df["account_type"] == account_type
    ].copy()

    # Need actual examples to generate from
    if class_data.empty:
        continue

    print(
        f"{account_type}: "
        f"{count} original -> "
        f"{deficit} synthetic"
    )

    existing_names = class_data["account_name"].astype(str).tolist()
    existing_ids = set(combined_df["account_id"].astype(str))

    generated_names = set()

    attempts = 0
    max_attempts = deficit * 20

    while len(generated_names) < deficit and attempts < max_attempts:

        attempts += 1

        # Select a real example from this class
        source_name = random.choice(existing_names)

        synthetic_name = generate_synthetic_account_name(
            source_name,
            account_type
        )

        # Avoid duplicates
        if synthetic_name in existing_names:
            continue

        if synthetic_name in generated_names:
            continue

        generated_names.add(synthetic_name)

    # ---------------------------------------------------------
    # Create synthetic rows.
    # ---------------------------------------------------------

    for i, synthetic_name in enumerate(generated_names):

        synthetic_id = (
            f"synthetic_{account_type}_{i:05d}"
        )

        synthetic_rows.append({
            "account_id": synthetic_id,
            "account_name": synthetic_name,
            "account_type": account_type,
            "country": "synthetic"
        })


# ---------------------------------------------------------
# Add synthetic data to original data.
# ---------------------------------------------------------

if synthetic_rows:

    synthetic_df = pd.DataFrame(synthetic_rows)

    combined_df = pd.concat(
        [combined_df, synthetic_df],
        ignore_index=True
    )

else:

    synthetic_df = pd.DataFrame(
        columns=combined_df.columns
    )


# ---------------------------------------------------------
# Verify class balance.
# ---------------------------------------------------------

final_counts = combined_df["account_type"].value_counts()

print("\nSynthetic rows generated:", len(synthetic_df))

print("\nClass counts AFTER synthetic generation:")
print(final_counts.sort_index())

print(
    "\nClasses still below target:",
    final_counts[final_counts < TARGET_INSTANCES]
)

print(
    f"\nTotal dataset after synthetic generation: "
    f"{len(combined_df)} rows"
)"""

In [ ]:
# ---------------------------------------------------------
# 4. Drop account_types with 200 or fewer instances
# ---------------------------------------------------------
MIN_INSTANCES = 590
class_counts = combined_df["account_type"].value_counts()
valid_classes = class_counts[class_counts > MIN_INSTANCES].index
dropped_classes = class_counts[class_counts <= MIN_INSTANCES]

if not dropped_classes.empty:
    print(f"Dropping {len(dropped_classes)} classes with <= {MIN_INSTANCES} instances:")
    print(dropped_classes)

capped_df = combined_df[combined_df["account_type"].isin(valid_classes)].reset_index(drop=True)
print(f"\nCombined (post-filter, pre-cutoff): {len(capped_df)} rows, {capped_df['account_type'].nunique()} classes")


# ---------------------------------------------------------
# 4. Cap each account_type at 500 records, globally across countries
# ---------------------------------------------------------
CUTOFF = 1000 # isko change krskte hain but biasness na aajaye
capped_parts = []
for account_type, group in capped_df.groupby("account_type"):
    capped_parts.append(group.sample(n=min(len(group), CUTOFF), random_state=42))
capped_df = pd.concat(capped_parts, ignore_index=True)

print(f"Combined (post-cutoff): {len(capped_df)} rows")
print("\nPost-cutoff account_type breakdown:\n", capped_df["account_type"].value_counts())


Combined (post-filter, pre-cutoff): 23053 rows, 18 classes
Combined (post-cutoff): 13963 rows

Post-cutoff account_type breakdown:
 account_type
asset_current            1000
asset_fixed              1000
liability_current        1000
asset_non_current        1000
expense                  1000
income                   1000
equity                    991
income_other              856
liability_non_current     722
liability_payable         600
asset_receivable          600
expense_depreciation      600
equity_unaffected         600
asset_cash                599
asset_prepayments         599
expense_direct_cost       599
liability_credit_card     599
off_balance               598
Name: count, dtype: int64


In [ ]:
import numpy as np

def build_train_test(combined_df, test_size=0.2, random_state=42):
    train_parts, test_parts = [], []
    flagged = []

    for acc_type, group in combined_df.groupby("account_type"):
        real = group[group["is_synthetic"] == False]
        synth = group[group["is_synthetic"] == True]
        total = len(group)
        test_target = round(test_size * total)

        if len(real) >= test_target:
            test_real = real.sample(n=test_target, random_state=random_state)
            train_real = real.drop(test_real.index)

            train_parts.append(train_real)
            train_parts.append(synth)
            test_parts.append(test_real)

        else:
            # not enough real rows to fill the test quota alone
            deficit = min(test_target - len(real), len(synth))
            test_synth = synth.sample(n=deficit, random_state=random_state) if deficit > 0 else synth.iloc[0:0]
            train_synth = synth.drop(test_synth.index)

            test_parts.append(real)          # all real goes to test
            test_parts.append(test_synth)
            train_parts.append(train_synth)  # leftover synthetic goes to train

            if deficit > 0:
                flagged.append((acc_type, len(real), test_target, deficit))

    train_df = pd.concat(train_parts).sample(frac=1, random_state=random_state).reset_index(drop=True)
    test_df = pd.concat(test_parts).sample(frac=1, random_state=random_state).reset_index(drop=True)

    if flagged:
        print("⚠️ Classes where real data couldn't fill the test quota — synthetic added to test:")
        for acc_type, real_n, target, synth_used in flagged:
            print(f"  {acc_type}: {real_n} real available, needed {target} for test, added {synth_used} synthetic")

    return train_df, test_df

train_df, test_df = build_train_test(capped_df, test_size=0.2, random_state=42)
train_df.to_csv("/content/data/train.csv", index=False)
test_df.to_csv("/content/data/test.csv", index=False)

⚠️ Classes where real data couldn't fill the test quota — synthetic added to test:
  equity_unaffected: 29 real available, needed 120 for test, added 91 synthetic
  liability_credit_card: 1 real available, needed 120 for test, added 119 synthetic


In [ ]:
import shutil
from google.colab import files

shutil.make_archive("/content/data", 'zip', "/content/data")
files.download("/content/data.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
end_time = time.time()
duration = end_time - start_time

print(f"Data Preparation Duration: {duration:.2f} seconds")

In [ ]:
# !unzip "data (1).zip" -d "/content/data"

# Training and Evaluation

In [ ]:
!pip install -q transformers scikit-learn pandas torch

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report
import pandas as pd
import numpy as np
import random
import json
import os

# ---------------------------------------------------------
# Config for this run — change these between experiments
# ---------------------------------------------------------
NUM_FROZEN_LAYERS = 0
DROPOUT = 0.1
EPOCHS = 20

# ---------------------------------------------------------
# Reproducibility + device
# ---------------------------------------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ---------------------------------------------------------
# Load the combined train/test CSVs saved from notebook 1
# ---------------------------------------------------------
train_df = pd.read_csv("/content/data/train.csv")
test_df = pd.read_csv("/content/data/test.csv")

type_encoder = LabelEncoder().fit(pd.concat([train_df["account_type"], test_df["account_type"]]))
train_df["account_type_label"] = type_encoder.transform(train_df["account_type"])
test_df["account_type_label"] = type_encoder.transform(test_df["account_type"])

num_classes = len(type_encoder.classes_)
print("Classes:", list(type_encoder.classes_))

# ---------------------------------------------------------
# Tokenizer + base model
# ---------------------------------------------------------
model_name = "sentence-transformers/all-MiniLM-L6-v2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
base_model = AutoModel.from_pretrained(model_name).to(device)

"""
# Freeze embeddings
for param in base_model.embeddings.parameters():
    param.requires_grad = False

# Freeze first of MiniLM's 6 encoder layers
for layer in base_model.encoder.layer[:(NUM_FROZEN_LAYERS-1)]:
    for param in layer.parameters():
        param.requires_grad = False
"""

# ---------------------------------------------------------
# Dataset — now uses account_type_label (int) instead of raw string labels
# ---------------------------------------------------------
class AccountNameDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=32):
        self.names = df["account_name"].tolist()
        self.labels = df["account_type_label"].tolist()
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.names)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.names[idx],
            padding="max_length",
            truncation=True,
            max_length=self.max_len,
            return_tensors="pt",
        )
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "label": torch.tensor(self.labels[idx], dtype=torch.long),
        }

train_dataset = AccountNameDataset(train_df, tokenizer)
test_dataset = AccountNameDataset(test_df, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16)

# ---------------------------------------------------------
# Model: MiniLM encoder + classification head (mean pooling)
# ---------------------------------------------------------
def mean_pool(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    summed = torch.sum(last_hidden_state * mask, dim=1)
    counts = torch.clamp(mask.sum(dim=1), min=1e-9)
    return summed / counts

class AccountTypeClassifier(nn.Module):
    def __init__(self, base_model, hidden_size=384, num_classes=5):
        super().__init__()
        self.base_model = base_model
        self.dropout = nn.Dropout(DROPOUT)
        self.classifier = nn.Linear(hidden_size, num_classes)

    def forward(self, input_ids, attention_mask):
        outputs = self.base_model(input_ids=input_ids, attention_mask=attention_mask)
        pooled = mean_pool(outputs.last_hidden_state, attention_mask)
        pooled = self.dropout(pooled)
        return self.classifier(pooled)

model = AccountTypeClassifier(base_model, num_classes=num_classes).to(device)

# ---------------------------------------------------------
# Training
# ---------------------------------------------------------
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
loss_fn = nn.CrossEntropyLoss()

def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)
            logits = model(input_ids, attention_mask)
            preds = torch.argmax(logits, dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
            all_preds.extend(preds.cpu().tolist())
            all_labels.extend(labels.cpu().tolist())
    return correct / total, all_preds, all_labels

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0
    for batch in train_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        optimizer.zero_grad()
        logits = model(input_ids, attention_mask)
        loss = loss_fn(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    train_acc, _, _ = evaluate(model, train_loader)
    test_acc, _, _ = evaluate(model, test_loader)
    print(f"Epoch {epoch:2d} | loss: {total_loss/len(train_loader):.4f} | train_acc: {train_acc:.3f} | test_acc: {test_acc:.3f}")

# ---------------------------------------------------------
# Final test report (per-class — worth checking given class imbalance)
# ---------------------------------------------------------
import time

start_time = time.time()

final_test_acc, test_preds, test_labels = evaluate(model, test_loader)

end_time = time.time()
duration = end_time - start_time

print(f"Evaluation Duration: {duration:.2f} seconds")
print(f"\nFinal test accuracy: {final_test_acc:.3f}\n")
print(classification_report(
    test_labels, test_preds,
    labels=list(range(num_classes)),
    target_names=type_encoder.classes_,
    zero_division=0,
))

Using device: cpu
Classes: ['asset_cash', 'asset_current', 'asset_fixed', 'asset_non_current', 'asset_prepayments', 'asset_receivable', 'equity', 'equity_unaffected', 'expense', 'expense_depreciation', 'expense_direct_cost', 'income', 'income_other', 'liability_credit_card', 'liability_current', 'liability_non_current', 'liability_payable', 'off_balance']


config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Epoch  1 | loss: 1.8788 | train_acc: 0.597 | test_acc: 0.499
Epoch  2 | loss: 1.2543 | train_acc: 0.685 | test_acc: 0.552
Epoch  3 | loss: 1.0463 | train_acc: 0.743 | test_acc: 0.586
Epoch  4 | loss: 0.9002 | train_acc: 0.784 | test_acc: 0.587
Epoch  5 | loss: 0.7880 | train_acc: 0.824 | test_acc: 0.604
Epoch  6 | loss: 0.6936 | train_acc: 0.855 | test_acc: 0.602
Epoch  7 | loss: 0.6141 | train_acc: 0.880 | test_acc: 0.616
Epoch  8 | loss: 0.5477 | train_acc: 0.900 | test_acc: 0.624
Epoch  9 | loss: 0.4801 | train_acc: 0.914 | test_acc: 0.618
Epoch 10 | loss: 0.4338 | train_acc: 0.928 | test_acc: 0.624
Epoch 11 | loss: 0.3953 | train_acc: 0.936 | test_acc: 0.630
Epoch 12 | loss: 0.3533 | train_acc: 0.943 | test_acc: 0.626
Epoch 13 | loss: 0.3177 | train_acc: 0.950 | test_acc: 0.630
Epoch 14 | loss: 0.2921 | train_acc: 0.956 | test_acc: 0.619
Epoch 15 | loss: 0.2810 | train_acc: 0.960 | test_acc: 0.627
Epoch 16 | loss: 0.2545 | train_acc: 0.960 | test_acc: 0.627
Epoch 17 | loss: 0.2389 

Logging for experimentation

In [ ]:
# ---------------------------------------------------------
# Log this run's config + metrics to a persistent Excel file
# ---------------------------------------------------------
import os
from datetime import datetime

def log_run_to_excel(
    frozen_layers,
    dropout,
    final_test_acc,
    test_preds,
    test_labels,
    type_encoder,
    min_instances,
    excel_path="/content/experiment_log.xlsx",
    extra_info=None,
):
    report_dict = classification_report(
        test_labels, test_preds,
        labels=list(range(len(type_encoder.classes_))),
        target_names=type_encoder.classes_,
        zero_division=0,
        output_dict=True,
    )

    row = {
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "frozen_layers": frozen_layers,
        "dropout": dropout,
        "min_instances": min_instances,
        "test_accuracy": final_test_acc,
        "macro_precision": report_dict["macro avg"]["precision"],
        "macro_recall": report_dict["macro avg"]["recall"],
        "macro_f1": report_dict["macro avg"]["f1-score"],
        "weighted_precision": report_dict["weighted avg"]["precision"],
        "weighted_recall": report_dict["weighted avg"]["recall"],
        "weighted_f1": report_dict["weighted avg"]["f1-score"],
    }

    if extra_info:
        row.update(extra_info)

    # per-class F1 + support, so you can see exactly which classes
    # improve/degrade as you sweep frozen_layers / dropout
    for cls in type_encoder.classes_:
        row[f"f1_{cls}"] = report_dict[cls]["f1-score"]
        row[f"support_{cls}"] = report_dict[cls]["support"]

    new_row_df = pd.DataFrame([row])

    if os.path.exists(excel_path):
        existing_df = pd.read_excel(excel_path)
        combined_df = pd.concat([existing_df, new_row_df], ignore_index=True)
    else:
        combined_df = new_row_df

    combined_df.to_excel(excel_path, index=False)
    print(f"Logged run to {excel_path} (frozen_layers={frozen_layers}, dropout={dropout})")
    return combined_df

log_run_to_excel(
    frozen_layers=NUM_FROZEN_LAYERS,
    dropout=DROPOUT,
    final_test_acc=final_test_acc,
    test_preds=test_preds,
    test_labels=test_labels,
    type_encoder=type_encoder,
    min_instances=MIN_INSTANCES,   # set this to match the cutoff used when building train.csv/test.csv for this run
    extra_info={"epochs": EPOCHS, "lr": 2e-5, "train_acc_final": train_acc},
)

Logged run to /content/experiment_log.xlsx (frozen_layers=0, dropout=0.1)


,timestamp,frozen_layers,dropout,min_instances,test_accuracy,macro_precision,macro_recall,macro_f1,weighted_precision,weighted_recall,...,f1_liability_credit_card,support_liability_credit_card,f1_liability_current,support_liability_current,f1_liability_non_current,support_liability_non_current,f1_liability_payable,support_liability_payable,f1_off_balance,support_off_balance
0,2026-09-05 19:56:30,0,0.1,590,0.62406,0.657171,0.62903,0.633174,0.639151,0.62406,...,0.987448,120.0,0.492754,200.0,0.64539,144.0,0.642336,120.0,0.573991,120.0


In [ ]:
from google.colab import files

files.download("/content/experiment_log.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Finetuned Model Save for Later Use

In [ ]:
# ---------------------------------------------------------
# Save fine-tuned model + tokenizer + label encoder
# ---------------------------------------------------------
save_dir = "/content/account_type_classifier"
os.makedirs(save_dir, exist_ok=True)

torch.save(model.state_dict(), os.path.join(save_dir, "model_state_dict.pt"))
tokenizer.save_pretrained(save_dir)

with open(os.path.join(save_dir, "label_classes.json"), "w") as f:
    json.dump(list(type_encoder.classes_), f)

with open(os.path.join(save_dir, "config.json"), "w") as f:
    json.dump({"base_model": model_name, "hidden_size": 384, "num_classes": num_classes, "max_len": 32}, f)

print(f"\nSaved model, tokenizer, and label classes to {save_dir}")

# Create final COA

In [ ]:
IFRS_LOOKUP_TABLE = {

    # ============================================================
    # 1. STATEMENT OF FINANCIAL POSITION
    # ============================================================

    "asset": "Statement of Financial Position",
    "asset_current": "Statement of Financial Position",
    "asset_non_current": "Statement of Financial Position",
    "asset_fixed": "Statement of Financial Position",
    "asset_receivable": "Statement of Financial Position",
    "asset_cash": "Statement of Financial Position",
    "asset_prepayments": "Statement of Financial Position",

    "liability": "Statement of Financial Position",
    "liability_current": "Statement of Financial Position",
    "liability_non_current": "Statement of Financial Position",
    "liability_payable": "Statement of Financial Position",
    "liability_credit_card": "Statement of Financial Position",

    "equity": "Statement of Financial Position",
    "equity_unaffected": "Statement of Financial Position",

    # Common IFRS position subcategories
    "cash_and_cash_equivalents": "Statement of Financial Position",
    "trade_and_other_receivables": "Statement of Financial Position",
    "inventories": "Statement of Financial Position",
    "current_tax_assets": "Statement of Financial Position",
    "other_current_assets": "Statement of Financial Position",
    "property_plant_and_equipment": "Statement of Financial Position",
    "investment_property": "Statement of Financial Position",
    "intangible_assets": "Statement of Financial Position",
    "right_of_use_assets": "Statement of Financial Position",
    "financial_assets": "Statement of Financial Position",
    "deferred_tax_assets": "Statement of Financial Position",
    "other_non_current_assets": "Statement of Financial Position",

    "trade_and_other_payables": "Statement of Financial Position",
    "current_tax_liabilities": "Statement of Financial Position",
    "provisions_current": "Statement of Financial Position",
    "borrowings_current": "Statement of Financial Position",
    "lease_liabilities_current": "Statement of Financial Position",
    "other_current_liabilities": "Statement of Financial Position",
    "borrowings_non_current": "Statement of Financial Position",
    "lease_liabilities_non_current": "Statement of Financial Position",
    "deferred_tax_liabilities": "Statement of Financial Position",
    "provisions_non_current": "Statement of Financial Position",
    "other_non_current_liabilities": "Statement of Financial Position",

    "issued_capital": "Statement of Financial Position",
    "share_capital": "Statement of Financial Position",
    "treasury_shares": "Statement of Financial Position",
    "retained_earnings": "Statement of Financial Position",
    "other_reserves": "Statement of Financial Position",
    "non_controlling_interests": "Statement of Financial Position",


    # ============================================================
    # 2. STATEMENT OF PROFIT OR LOSS
    # ============================================================

    "income": "Statement of Profit or Loss",
    "income_other": "Statement of Profit or Loss",
    "revenue": "Statement of Profit or Loss",
    "operating_revenue": "Statement of Profit or Loss",
    "other_income": "Statement of Profit or Loss",

    "expense": "Statement of Profit or Loss",
    "expense_direct_cost": "Statement of Profit or Loss",
    "expense_depreciation": "Statement of Profit or Loss",
    "cost_of_sales": "Statement of Profit or Loss",
    "administrative_expenses": "Statement of Profit or Loss",
    "selling_expenses": "Statement of Profit or Loss",
    "distribution_expenses": "Statement of Profit or Loss",
    "employee_benefits_expense": "Statement of Profit or Loss",
    "depreciation_expense": "Statement of Profit or Loss",
    "amortisation_expense": "Statement of Profit or Loss",
    "finance_costs": "Statement of Profit or Loss",
    "income_tax_expense": "Statement of Profit or Loss",
    "impairment_expense": "Statement of Profit or Loss",
    "foreign_exchange_gain_loss": "Statement of Profit or Loss",


    # ============================================================
    # 3. OTHER COMPREHENSIVE INCOME (OCI)
    # ============================================================

    "oci": "Statement of Other Comprehensive Income",
    "other_comprehensive_income": "Statement of Other Comprehensive Income",
    "foreign_currency_translation": "Statement of Other Comprehensive Income",
    "revaluation_surplus": "Statement of Other Comprehensive Income",
    "fair_value_changes_oci": "Statement of Other Comprehensive Income",
    "cash_flow_hedge_reserve": "Statement of Other Comprehensive Income",
    "remeasurements_employee_benefits": "Statement of Other Comprehensive Income",


    # ============================================================
    # 4. STATEMENT OF CHANGES IN EQUITY
    # ============================================================

    "changes_in_equity": "Statement of Changes in Equity",
    "share_capital_changes": "Statement of Changes in Equity",
    "share_premium_changes": "Statement of Changes in Equity",
    "retained_earnings_changes": "Statement of Changes in Equity",
    "reserve_changes": "Statement of Changes in Equity",
    "dividends": "Statement of Changes in Equity",
    "transactions_with_owners": "Statement of Changes in Equity",
    "total_comprehensive_income": "Statement of Changes in Equity",
    "non_controlling_interest_changes": "Statement of Changes in Equity",


    # ============================================================
    # 5. STATEMENT OF CASH FLOWS
    # ============================================================

    "cash_flow": "Statement of Cash Flows",

    # Operating activities
    "cash_flows_operating": "Statement of Cash Flows",
    "cash_received_from_customers": "Statement of Cash Flows",
    "cash_paid_to_suppliers": "Statement of Cash Flows",
    "cash_paid_to_employees": "Statement of Cash Flows",
    "cash_paid_for_operating_expenses": "Statement of Cash Flows",
    "income_taxes_paid": "Statement of Cash Flows",

    # Investing activities
    "cash_flows_investing": "Statement of Cash Flows",
    "purchase_of_property_plant_equipment": "Statement of Cash Flows",
    "proceeds_from_sale_of_property_plant_equipment": "Statement of Cash Flows",
    "purchase_of_investments": "Statement of Cash Flows",
    "proceeds_from_sale_of_investments": "Statement of Cash Flows",

    # Financing activities
    "cash_flows_financing": "Statement of Cash Flows",
    "proceeds_from_borrowings": "Statement of Cash Flows",
    "repayment_of_borrowings": "Statement of Cash Flows",
    "proceeds_from_issue_of_shares": "Statement of Cash Flows",
    "dividends_paid": "Statement of Cash Flows",
    "lease_payments": "Statement of Cash Flows",


    # ============================================================
    # 6. NOTES / DISCLOSURES
    # ============================================================

    "notes": "Notes to Financial Statements",

    "accounting_policies": "Notes to Financial Statements",
    "corporate_information": "Notes to Financial Statements",
    "ifrs_compliance": "Notes to Financial Statements",
    "property_plant_equipment_notes": "Notes to Financial Statements",
    "financial_instruments_notes": "Notes to Financial Statements",
    "fair_value_notes": "Notes to Financial Statements",
    "intangible_assets_notes": "Notes to Financial Statements",
    "investment_property_notes": "Notes to Financial Statements",
    "inventory_notes": "Notes to Financial Statements",
    "provisions_notes": "Notes to Financial Statements",
    "contingent_liabilities": "Notes to Financial Statements",
    "revenue_notes": "Notes to Financial Statements",
    "government_grants_notes": "Notes to Financial Statements",
    "impairment_notes": "Notes to Financial Statements",
    "leases_notes": "Notes to Financial Statements",
    "share_based_payment_notes": "Notes to Financial Statements",
    "employee_benefits_notes": "Notes to Financial Statements",
    "income_tax_notes": "Notes to Financial Statements",
    "borrowing_costs_notes": "Notes to Financial Statements",
    "insurance_contract_notes": "Notes to Financial Statements",
    "earnings_per_share_notes": "Notes to Financial Statements",
    "related_party_notes": "Notes to Financial Statements",
    "business_combination_notes": "Notes to Financial Statements",
    "events_after_reporting_period_notes": "Notes to Financial Statements",
}

In [ ]:
import os
import json
import torch
import torch.nn as nn
import pandas as pd
from glob import glob
from transformers import AutoTokenizer, AutoModel

# ---------------------------------------------------------
# Config — adjust these
# ---------------------------------------------------------
MODEL_DIR = "/content/account_type_classifier"
TRIAL_BALANCE_FOLDER = "/content/trial_balances"
OUTPUT_PATH = "/content/chart_of_accounts.xlsx"
SKIPPED_PATH = "/content/skipped_duplicates.xlsx"
ACCOUNT_NAME_COL = "Account Name"     # <- change to match your file's actual column
ACCOUNT_ID_COL = "Account ID"          # <- change or set to None if not present


PARENT_CATEGORY_MAP = {
    "asset_cash": "asset", "asset_current": "asset", "asset_fixed": "asset",
    "asset_non_current": "asset", "asset_prepayments": "asset", "asset_receivable": "asset",
    "equity": "equity", "equity_unaffected": "equity",
    "expense": "expense", "expense_depreciation": "expense", "expense_direct_cost": "expense",
    "income": "income", "income_other": "income",
    "liability_credit_card": "liability", "liability_current": "liability",
    "liability_non_current": "liability", "liability_payable": "liability",
    "off_balance": None,
}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---------------------------------------------------------
# Load fine-tuned model (once, at module load)
# ---------------------------------------------------------
with open(os.path.join(MODEL_DIR, "config.json")) as f:
    model_cfg = json.load(f)
with open(os.path.join(MODEL_DIR, "label_classes.json")) as f:
    label_classes = json.load(f)

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
base_model = AutoModel.from_pretrained(model_cfg["base_model"]).to(device)

def mean_pool(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    summed = torch.sum(last_hidden_state * mask, dim=1)
    counts = torch.clamp(mask.sum(dim=1), min=1e-9)
    return summed / counts

class AccountTypeClassifier(nn.Module):
    def __init__(self, base_model, hidden_size, num_classes):
        super().__init__()
        self.base_model = base_model
        self.classifier = nn.Linear(hidden_size, num_classes)

    def forward(self, input_ids, attention_mask):
        outputs = self.base_model(input_ids=input_ids, attention_mask=attention_mask)
        pooled = mean_pool(outputs.last_hidden_state, attention_mask)
        return self.classifier(pooled)

model = AccountTypeClassifier(base_model, model_cfg["hidden_size"], model_cfg["num_classes"]).to(device)
model.load_state_dict(torch.load(os.path.join(MODEL_DIR, "model_state_dict.pt"), map_location=device))
model.eval()

def predict_account_type(account_name: str) -> str:
    enc = tokenizer(
        account_name, padding="max_length", truncation=True,
        max_length=model_cfg["max_len"], return_tensors="pt",
    ).to(device)
    with torch.no_grad():
        logits = model(enc["input_ids"], enc["attention_mask"])
        pred_idx = torch.argmax(logits, dim=1).item()
    return label_classes[pred_idx]

# ---------------------------------------------------------
# Load existing chart of accounts once, to check duplicates against
# ---------------------------------------------------------
def load_existing_account_names(output_path: str) -> set:
    if os.path.exists(output_path):
        existing_df = pd.read_excel(output_path)
        if "account_name" in existing_df.columns:
            return set(existing_df["account_name"].astype(str).str.strip())
    return set()

# ---------------------------------------------------------
# Process a single trial balance file
# Returns (new_rows, skipped_rows) — does not write to disk itself
# ---------------------------------------------------------
def process_file(filepath: str, seen_account_names: set) -> tuple[list, list]:
    df = pd.read_excel(filepath)

    if ACCOUNT_NAME_COL not in df.columns:
        raise KeyError(
            f"'{ACCOUNT_NAME_COL}' not found in {os.path.basename(filepath)}. "
            f"Available columns: {list(df.columns)}"
        )

    new_rows, skipped_rows = [], []

    for idx, row in df.iterrows():
        account_name = str(row[ACCOUNT_NAME_COL]).strip()
        if not account_name or account_name.lower() == "nan":
            continue

        account_id = row[ACCOUNT_ID_COL] if ACCOUNT_ID_COL and ACCOUNT_ID_COL in df.columns else idx

        if account_name in seen_account_names:
            skipped_rows.append({
                "account_id": account_id,
                "account_name": account_name,
                "source_file": os.path.basename(filepath),
                "reason": "duplicate_account_name",
            })
            continue

        subtype = predict_account_type(account_name)
        parent_category = PARENT_CATEGORY_MAP.get(subtype, None)
        financial_statement = IFRS_LOOKUP_TABLE.get(subtype, None)

        new_rows.append({
            "account_id": account_id,
            "account_name": account_name,
            "account_type": parent_category,
            "subtype": subtype,
            "financial_statement": financial_statement,
            "source_file": os.path.basename(filepath),
        })

        seen_account_names.add(account_name)   # so duplicates within/across files in this run are also caught

    return new_rows, skipped_rows

# ---------------------------------------------------------
# Process an entire folder — finds files, delegates each to process_file
# ---------------------------------------------------------
def process_folder(folder_path: str, output_path: str = OUTPUT_PATH, skipped_path: str = SKIPPED_PATH):
    files = glob(os.path.join(folder_path, "*.xls*"))
    if not files:
        raise FileNotFoundError(f"No Excel files found in {folder_path}")

    seen_account_names = load_existing_account_names(output_path)

    all_new_rows, all_skipped_rows = [], []

    for filepath in files:
        new_rows, skipped_rows = process_file(filepath, seen_account_names)
        all_new_rows.extend(new_rows)
        all_skipped_rows.extend(skipped_rows)

    # Write chart of accounts (append if exists)
    if all_new_rows:
        new_df = pd.DataFrame(all_new_rows)
        if os.path.exists(output_path):
            existing_df = pd.read_excel(output_path)
            combined_df = pd.concat([existing_df, new_df], ignore_index=True)
        else:
            combined_df = new_df
        combined_df.to_excel(output_path, index=False)

    # Write skipped duplicates (append if exists)
    if all_skipped_rows:
        skipped_df = pd.DataFrame(all_skipped_rows)
        if os.path.exists(skipped_path):
            existing_skipped_df = pd.read_excel(skipped_path)
            combined_skipped_df = pd.concat([existing_skipped_df, skipped_df], ignore_index=True)
        else:
            combined_skipped_df = skipped_df
        combined_skipped_df.to_excel(skipped_path, index=False)

    print(f"Logged {len(all_new_rows)} new accounts to {output_path}")
    print(f"Skipped {len(all_skipped_rows)} duplicate accounts, logged to {skipped_path}")